##导入相关包

In [ ]:
import pandas as pd
import numpy as np
import jieba  #jieba分词
import os   
from sklearn.feature_extraction.text import TfidfVectorizer #TF-IDF
from sklearn.model_selection import GridSearchCV            #网格搜索
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import confusion_matrix,classification_report #预测效果的评估
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif']=['SimHei']  #这两行用于plt图像显示中文，否则plt无法显示中文
plt.rcParams['axes.unicode_minus']=False

# 實驗1(dataset from nkust)

## 讀取檔案

In [ ]:
import json

# 開啟 JSON 檔案
with open('/content/drive/MyDrive/CollectEventData/nkust-topic.json', 'r') as f:
    # 解析 JSON 資料
    data = json.load(f)

# 顯示資料
print(data)

## 斷詞

In [ ]:
!pip install ckiptagger
!pip install ckipnlp
!pip install ckip_classic

In [ ]:
from ckipnlp.pipeline import CkipPipeline, CkipDocument
from ckipnlp.pipeline import CkipPipeline, CkipDocument

pipeline = CkipPipeline(opts={'con_parser': {'username': "infoamplify", 'password': "2223pj[DB]"}})

In [ ]:
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER
from ckipnlp.pipeline import CkipPipeline, CkipDocument
from ckipnlp.pipeline import CkipPipeline, CkipDocument
import json
newJsonFile=[]
# 載入 ckiptagger 中文斷詞模型
ws = WS("/content/drive/MyDrive/CKIP/data")

# 讀取 JSON 檔案
with open("/content/drive/MyDrive/CollectEventData/nkust-topic.json", "r") as f:
    data = json.load(f)

# 對 JSON 檔案內的文字進行斷詞
for item in data:
    # 取得文字內容
    title = item.get("title", None)
    text = item.get("content", None)
    # 將文字進行斷詞
    doc = CkipDocument(raw=title + text)
    pipeline.get_ws(doc)

    # word_list = ws(title + text, sentence_segmentation=True)
    word_list_str = ' '.join([line.to_text() for line in doc.ws])

    newJsonFile.append({
        "TitleAndDESC":word_list_str,
        "Topic":["教育活動"],
    })
# # 將新的 JSON 資料存入檔案中。
# with open('/content/drive/MyDrive/collab-topic/斷詞後的資料/From_nkust.txt', 'w', encoding='utf-8') as f:
#     json.dump(newJsonFile, f, ensure_ascii=False, indent=4)

## 特徵選取(TFIDF) 
用於把無意義的詞拿走


In [ ]:
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER
import json
from sklearn.feature_extraction.text import TfidfVectorizer
import re
# 載入 ckiptagger 中文斷詞模型
# ws = WS("/path/to/your/ckiptagger/data")

# 讀取 JSON 檔案
with open("/content/drive/MyDrive/collab-topic/斷詞後的資料/From_nkust.txt", "r") as f:
    data = json.load(f)

# 將每個檔案中的 TitleAndDESC 值轉換為文本
corpus = []
for item in data:
    title_and_desc = item.get("TitleAndDESC", [])
    text = "".join([w for sent in title_and_desc for w in sent])
    corpus.append(text)

# print('corpus：\n',corpus[2])
# 将数字和特殊字符从特征名称中删除
# pattern = r'[^a-zA-Z\u4e00-\u9fa5]+'
# # 匹配中文或英文字母开头的词语'
# pattern = r'^[\u4e00-\u9fa5a-zA-Z]+'
# 清理corpus，因為數字型字串，停用詞檔不了
# pattern = r'^[\u4e00-\u9fa5]+[\w]* |[^a-zA-Z\u4e00-\u9fa5]+'
# clean_corpus = []

# for text in corpus:
#     cleaned_text = re.sub(pattern, ' ', text)
#     clean_corpus.append(cleaned_text)

# print(clean_corpus)

# original_stop_words = ["，", "。", "；", "：", "“", "”", "（", "）", "《", "》", "、", "？", "「", "」", "『", "』", "‘", "’", "...", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "a", "an", "and", "the", "or", "but", "so"]
original_stop_words = ["，", "。", "；", "：", "“", "”", "（", "）", "《", "》", "、", "？", "「", "」", "『", "』", "‘", "’", "...", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
with open("/content/drive/MyDrive/中文新闻文本标题分类Data/stop_words.txt", "r", encoding="utf-8") as f:
  stop_words = f.read().split("\n")
stop_words += original_stop_words
# 只提取(至少一個)中文字，该正则表达式匹配的是连续的中文字符，其中\u4e00和\u9fa5分别代表Unicode编码中中文字符的开始和结束位置。
pattern = r'[\u4e00-\u9fa5]+'
vectorizer  = TfidfVectorizer(stop_words=stop_words, token_pattern=pattern)#me：這只是設定參數何以
X = vectorizer.fit_transform(corpus)#把設定套用在corpus

# 打印特徵選取結果
feature_names = vectorizer.get_feature_names()
# print('feature_names',feature_names)
# print('vectorizer',vectorizer)
# print('X：',X)

# for i, doc in enumerate(data):
#     print("File {}: ".format(i))
#     title_and_desc = doc.get("TitleAndDESC", [])
#     text = " ".join([w for sent in title_and_desc for w in sent])
#     feature_weights = X[i, :].toarray()[0]
#     sorted_indexes = feature_weights.argsort()[::-1]
#     for j in range(10):
#         feature_index = sorted_indexes[j]
#         print(feature_names[feature_index], feature_weights[feature_index])
#     print("\n")


## 特徵選取(主題模型)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# 訓練主題模型
lda = LatentDirichletAllocation(n_components=10, random_state=42)# me：這只是設定參數何以
lda.fit(X)# 把設定套用在X(詞頻矩陣)

# 打印主題的潛在關鍵字
for i, topic in enumerate(lda.components_):
    top_features_idx = topic.argsort()[:-100 - 1:-1]
    top_features = [feature_names[idx] for idx in top_features_idx]
    print(f"Topic {i}: {', '.join(top_features)}")

In [ ]:
for i, topic in enumerate(lda.show_topics(num_topics=1, num_words=10)):
  print('從教育活動資料集中的潛在主題關鍵字:{}'.format(topic))

X 是一個文檔集合（corpus）經過向量化（vectorization）處理後所得到的特徵矩陣。向量化通常是將文本表示為數字特徵的過程，例如文本中的詞頻、TF-IDF 值等。這裡使用的是 vectorizer.fit_transform() 方法將原始文檔轉換為特徵矩陣。

X 通常被稱為「特徵矩陣」或「訓練集」。在機器學習中，我們通常使用 X 來表示輸入特徵，也就是用來訓練模型的變量。因此，在這段程式碼中，X 可以稱為「主題模型的輸入特徵」或「訓練集」。

X变量通常称为词频矩阵，它是将文本数据转换为数字特征的常见表示形式之一，其中每个文档由一个向量表示，该向量表示在词汇表中每个词的出现次数或权重。主题模型通常使用词频矩阵作为输入数据。

X變數是詞頻-逆向文件頻率(TF-IDF)矩陣，用來表示文本集合中每個文檔中詞語的權重。

在TF-IDF矩陣中，每行代表一個文檔，每列代表一個詞彙，詞頻-逆向文件頻率權重用於衡量該詞彙對文檔的重要性。每個元素的值是在文檔中詞彙出現次數的基礎上乘以它在整個文集中出現的頻率的倒數。

對於每個元素(i, j)，i表示文檔的索引，j表示詞彙的索引，其值代表文檔i中詞彙j的TF-IDF權重。例如，(0, 635) 0.016893641184136898表示第0個文檔(corpus中的第一個文檔)中詞彙635的TF-IDF權重為0.016893641184136898 = TF*IDF。




LatentDirichletAllocation 是一種主題模型，用於從文本文檔中學習主題並找到潛在的主題。以下是 LatentDirichletAllocation 的主要參數及其用法：

1. n_components: 模型中主題的数量。
2. doc_topic_prior: 每个文档分配给每个主题的先验分布。可以是浮点数或数组。当 alpha 为 None 时，将其设置为 1.0 / n_components。
3. topic_word_prior: 每个主题分配给每个单词的先验分布。可以是浮点数或数组。当 beta 为 None 时，将其设置为 1.0 / n_components。
4. learning_method: 用于拟合模型的学习方法。'batch' 用于在线变分推断，'online' 用于小批量变分推断。 默认为 'batch'。
5. learning_decay: 用于控制在线变分推断更新速度的学习率。 当学习方法为 'online' 时使用。默认为 0.7。
6. learning_offset: 用于控制在线变分推断的学习率。 默认为 10.0。
7. max_iter: 迭代的最大次数。默认为 10。
8. batch_size: 在线学习时，每个小批次的文档数。
9. evaluate_every: 在训练过程中进行对数似然损失评估的频率。
10. total_samples: 对于在线变分推断，文档的总数。默认为 1e6。
11. verbose: 是否输出详细日志信息。默认为 0。
12. random_state: 随机数生成器的种子。默认为 None。

以下是 LatentDirichletAllocation 的常用方法：

1. fit: 训练模型，返回模型对象。
2. fit_transform: 训练模型并转换数据，返回文档-主题矩阵。
3. transform: 转换数据，返回文档-主题矩阵。
4. score: 返回数据的对数似然损失。
5. perplexity: 返回数据的困惑度。

# 實驗2(dataset from nuk)

## 讀取檔案

1. 先利用 Python 內建的 os 模組取得這215個json檔案的完整路徑，並存入一個列表中。用 os.path.join() 函式結合路徑。
2. 讀取這215個json檔案，並將它們的內容合併到一個新的json檔案中。

In [ ]:
import os

nuk_dir = "/content/drive/MyDrive/collab-topic/斷詞前的資料/nuk"  # nuk目錄的路徑
json_files = []
for i in range(1, 216):
    json_file = os.path.join(nuk_dir, f"{i}.json")
    json_files.append(json_file)

import json

merged_json = []
for json_file in json_files:
    with open(json_file, "r") as f:
        json_data = json.load(f)
        merged_json.append(json_data)

# 顯示資料
print(merged_json)

# 將合併後的json資料寫入檔案//encode='utf-8' not set
with open("/content/drive/MyDrive/collab-topic/斷詞前的資料/nuk.txt", "w", encoding='utf-8') as f:
    json.dump(merged_json, f)

斷詞後nuk可以(格式ok)
## 斷詞


In [ ]:
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER
from ckipnlp.pipeline import CkipPipeline, CkipDocument
from ckipnlp.pipeline import CkipPipeline, CkipDocument
import json
newJsonFile=[]
# 載入 ckiptagger 中文斷詞模型
ws = WS("/content/drive/MyDrive/CKIP/data")

# 讀取 JSON 檔案
with open("/content/drive/MyDrive/collab-topic/斷詞前的資料/已有斷詞_程式不碰/nuk.json", "r") as f:
    data = json.load(f)

# 對 JSON 檔案內的文字進行斷詞
for item in data:
    # # 取得文字內容
    # title = item.get("title", None)
    # text = item.get("content", None)

    text = ' '.join(item.get("title", []))
    text = ' '.join(item.get("content", []))

    # 將文字進行斷詞
    doc = CkipDocument(raw=text)
    pipeline.get_ws(doc)

    # word_list = ws(title + text, sentence_segmentation=True)
    # word_list_str = '|'.join([line.to_text() for line in doc.ws])
    word_list_str =' '.join([line.to_text() for line in doc.ws])

    newJsonFile.append({
        "TitleAndDESC":word_list_str,
        "Topic":["教育活動"],
    })
# 將新的 JSON 資料存入檔案中。
with open('/content/drive/MyDrive/collab-topic/斷詞後的資料/nuk3.json', 'w', encoding='utf-8') as f:
    json.dump(newJsonFile, f, ensure_ascii=False, indent=4)

## 特徵選取(TFIDF)
用於把無意義的詞拿走
和提取重要的詞

In [ ]:
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER
import json
from sklearn.feature_extraction.text import TfidfVectorizer
import re

# 讀取 JSON 檔案
with open("/content/drive/MyDrive/collab-topic/斷詞後的資料/nuk.json", "r") as f:
    data = json.load(f)

# 將每個檔案中的 TitleAndDESC 值轉換為文本
corpus = []
for item in data:
    title_and_desc = item.get("TitleAndDESC", [])
    text = "".join([w for sent in title_and_desc for w in sent])
    corpus.append(text)

# original_stop_words = ["，", "。", "；", "：", "“", "”", "（", "）", "《", "》", "、", "？", "「", "」", "『", "』", "‘", "’", "...", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "a", "an", "and", "the", "or", "but", "so"]
original_stop_words = ["，", "。", "；", "：", "“", "”", "（", "）", "《", "》", "、", "？", "「", "」", "『", "』", "‘", "’", "...", "0", "1", "2", "3", "4", "5", "6", "7", "8", "9"]
with open("/content/drive/MyDrive/中文新闻文本标题分类Data/stop_words.txt", "r", encoding="utf-8") as f:
  stop_words = f.read().split("\n")
stop_words += original_stop_words
# 只提取(至少一個)中文字，该正则表达式匹配的是连续的中文字符，其中\u4e00和\u9fa5分别代表Unicode编码中中文字符的开始和结束位置。
pattern = r'[\u4e00-\u9fa5]+'
vectorizer  = TfidfVectorizer(stop_words=stop_words, token_pattern=pattern)#me：這只是設定參數何以
X = vectorizer.fit_transform(corpus)#把設定套用在corpus

# 打印特徵選取結果
feature_names = vectorizer.get_feature_names()
print('corpus',corpus)
print('feature_names',feature_names)
print('vectorizer',vectorizer)
print('X',X)


## 特徵選取(主題模型)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# 訓練主題模型
lda = LatentDirichletAllocation(n_components=10, random_state=42)# me：這只是設定參數何以
lda.fit(X)# 把設定套用在X(詞頻矩陣)

# 打印主題的潛在關鍵字
for i, topic in enumerate(lda.components_):
    top_features_idx = topic.argsort()[:-100 - 1:-1]
    top_features = [feature_names[idx] for idx in top_features_idx]
    print(f"Topic {i}: {', '.join(top_features)}")

In [ ]:
from ckiptagger import data_utils, construct_dictionary, WS, POS, NER
import json
from sklearn.feature_extraction.text import TfidfVectorizer
import re
# 載入 ckiptagger 中文斷詞模型
# ws = WS("/path/to/your/ckiptagger/data")

# 讀取 JSON 檔案
with open("/content/drive/MyDrive/collab-topic/斷詞後的資料/nuk.txt", "r") as f:
    data = json.load(f)

# 將每個檔案中的 TitleAndDESC 值轉換為文本
corpus = []
for item in data:
    title_and_desc = item.get("TitleAndDESC", [])
    # text = "".join([w for sent in title_and_desc for w in sent])
    corpus.append(title_and_desc)
print(corpus)

# nkust+nuk+開放政府資料 資瞭混和
## 讀取資料

In [ ]:
import json
import glob

# 取得資料夾內所有的 JSON 檔案路徑
json_files = glob.glob('/content/drive/MyDrive/collab-topic/斷詞前的資料/*.json')
merged_json = []
# 逐一讀取 JSON 檔案
for json_file in json_files:
    with open(json_file, 'r', encoding='utf-8') as f:
        json_data = json.load(f)
        for item in json_data:
   
            text = ' '.join(item.get("Title", []))
            text = ' '.join(item.get("Description", []))
            Topics = item.get("Topic", None)
            # 將文字進行斷詞
            doc = CkipDocument(raw=text)
            pipeline.get_ws(doc)
            word_list_str = ' '.join([line.to_text() for line in doc.ws])
            # 對Topics kry裡面放list or str 做處理
            if type(Topics) == str :# 裡面可能放多個
              Topics = [Topics]

            newJsonFile.append({
                    "TitleAndDESC":word_list_str,
                    "Topic":Topics,
                })
# 將新的 JSON 資料存入檔案中。
with open('/content/drive/MyDrive/collab-topic/斷詞後的資料/0222開放政府資料.json', 'w', encoding='utf-8') as f:
    json.dump(newJsonFile, f, ensure_ascii=False, indent=4)

用別人的會長什麼樣子

1. 对文本进行清洗及分词

In [ ]:
def clean(line):
    """对一个文件的数据进行清洗"""
    rep=['【】','【','】','👍','🤝',
        '🐮','🙏','🇨🇳','👏','❤️','………','🐰','...、、','，，','..','💪','🤓',
         '⚕️','👩','🙃','😇','🍺','🐂','🙌🏻','😂','📖','😭','✧٩(ˊωˋ*)و✧','🦐','？？？？','//','😊','💰','😜','😯',
         '(ღ˘⌣˘ღ)','✧＼٩(눈౪눈)و/／✧','🌎','🍀','🐴',
         '🌻','🌱','🌱','🌻','🙈','(ง•̀_•́)ง！','🉑️','💩',
         '🐎','⊙∀⊙！','🙊','【？','+1','😄','🙁','👇🏻','📚','🙇',
         '🙋','！！！！','🎉','＼(^▽^)／','👌','🆒','🏻',
         '🙉','🎵','🎈','🎊','0371-12345','☕️','🌞','😳','👻','🐶','👄','\U0001f92e\U0001f92e','😔','＋1','🛀','🐸','🐷','➕1',
         '🌚','：：','💉','√','x','！！！','🙅','♂️','💊','👋','o(^o^)o','mei\u2006sha\u2006shi','💉','😪','😱',
         '🤗','关注','……','(((╹д╹;)))','⚠️','Ծ‸Ծ','⛽️','😓','🐵',
         '🙄️','🌕','…','😋','[]','[',']','→_→','💞','😨','&quot;','😁','ฅ۶•ﻌ•♡','😰','🎙️',
         '🤧','😫','(ง•̀_•́)ง','😁','✊','🚬','😤','👻','😣','：','😷','(*^▽^)/★*☆','🐁','🐔','😘','🍋','(✪▽✪)','(❁´ω`❁)','1⃣3⃣','(^_^)／','☀️',
	     '🎁','😅','🌹','🏠','→_→','🙂','✨','❄️','•','🌤','💓','🔨','👏','😏','⊙∀⊙！','👍','✌(̿▀̿\u2009̿Ĺ̯̿̿▀̿̿)✌',
         '😊','👆','💤','😘','😊','😴','😉','🌟','♡♪..𝙜𝙤𝙤𝙙𝙣𝙞𝙜𝙝𝙩•͈ᴗ•͈✩‧₊˚','👪','💰','😎','🍀','🛍','🖕🏼','😂','(✪▽✪)','🍋','🍅','👀','♂️','🙋🏻','✌️','🥳','￣￣)σ',
         '😒','😉','🦀','💖','✊','💪','🙄','🎣','🌾','✔️','😡','😌','🔥','❤','🏼','🤭','🌿','丨','✅','🏥','ﾉ','☀','5⃣⏺1⃣0⃣','🚣','🎣','🤯','🌺',
         '🌸',
         ]
    pattern_0=re.compile('#.*?#')#在用户名处匹配话题名称
    pattern_1=re.compile('【.*?】')#在用户名处匹配话题名称
    pattern_2=re.compile('肺炎@([\u4e00-\u9fa5\w\-]+)')#匹配@
    pattern_3=re.compile('@([\u4e00-\u9fa5\w\-]+)')#匹配@
    #肺炎@环球时报
    pattern_4=re.compile(u'[\U00010000-\U0010ffff\uD800-\uDBFF\uDC00-\uDFFF]')#匹配表情
    pattern_5=re.compile('(.*?)')#匹配一部分颜文字
    pattern_7=re.compile('L.*?的微博视频')
    pattern_8=re.compile('（.*?）')
    #pattern_9=re.compile(u"\|[\u4e00-\u9fa5]*\|")#匹配中文

    line=line.replace('O网页链接','')
    line=line.replace('-----','')
    line=line.replace('①','')
    line=line.replace('②','')
    line=line.replace('③','')
    line=line.replace('④','')
    line=line.replace('>>','')
    line=re.sub(pattern_0, '', line,0) #去除话题
    line=re.sub(pattern_1, '', line,0) #去除【】
    line=re.sub(pattern_2, '', line,0) #去除@
    line=re.sub(pattern_3, '', line,0) #去除@
    line=re.sub(pattern_4, '', line,0) #去除表情
    line=re.sub(pattern_5, '', line,0) #去除一部分颜文字
    line=re.sub(pattern_7, '', line,0) 
    line=re.sub(pattern_8, '', line,0) 
    line=re.sub(r'\[\S+\]', '', line,0) #去除表情符号
    
    for i in rep:
        line=line.replace(i,'')
    return line

清洗完了之后，进行文本分词，代码如下：

In [ ]:
def seg_sentence(sentence):
    sentence = re.sub(u'[0-9\.]+', u'', sentence)
    jieba.load_userdict('自建词表.txt')#加载自建词表
    #suggest_freq((), tune=True) #修改词频，使其能分出来
    #jieba.add_word('知识集成')		# 这里是加入用户自定义的词来补充jieba词典
    sentence_seged =jieba.cut(sentence.strip(),cut_all=False,use_paddle=10)#默认精确模式
    #sentence_seged =jieba.cut_for_search(sentence.strip(),HMM=True)#搜索引擎模式
    #keywords =jieba.analyse.extract_tags(sentence, topK=30, withWeight=True, allowPOS=('n', 'v','nr', 'ns'))#关键词模式
    #sentence_seged=[item[0] for item in keywords]
    with open("/content/drive/MyDrive/中文新闻文本标题分类Data/stop_words.txt", "r", encoding="utf-8") as f:
        stopwords = f.read().split("\n")
    # stopwords = stopwordslist('/content/drive/MyDrive/collab-topic/斷詞後的資料/nuk.txt')  # 这里加载停用词的路径
    # synwords=synwordslist('近义词表.txt')#这里加载近义词的路径
    outstr = ''
    for word in sentence_seged:
        if word not in stopwords and word.__len__()>1:
            if word != '\t':#判断出不是停用词
            
                outstr += word
                outstr += " "
    return outstr

加载分词文件，构建词典及向量空间

In [ ]:
def infile(fliepath):
    #输入分词好的TXT，返回train
    train = []
    fp = open(fliepath,'r',encoding='utf8')
    for line in fp:
        new_line=[]
        if len(line)>1:
            line = line.strip().split(' ')
            for w in line:
                w.encode(encoding='utf-8')
                new_line.append(w)
        if len(new_line)>1:
            train.append(new_line)
    return train

3. 进行LDA分析

In [ ]:
def run(corpus_1,id2word_1,num,texts):
    #标准LDA算法
    lda_model = LdaModel(corpus=corpus_1, 
                         id2word=id2word_1,
                        num_topics=num,
                       passes=60,
                       alpha=(50/num),
                       eta=0.01,
                       random_state=42)
    # num_topics：主题数目
    # passes：训练伦次
    # num：每个主题下输出的term的数目
    #输出主题
    #topic_list = lda_model.print_topics()
    #for topic in topic_list:
        #print(topic)
    # 困惑度
    perplex=lda_model.log_perplexity(corpus_1)  # a measure of how good the model is. lower the better.
    # 一致性
    coherence_model_lda = CoherenceModel(model=lda_model, texts=texts, dictionary=id2word_1, coherence='c_v')
    coherence_lda = coherence_model_lda.get_coherence()
    #print('\n一致性指数: ', coherence_lda)   # 越高越好
    return lda_model,coherence_lda,perplex